In [36]:
import numpy as np
import torch

from pyscf import gto
from pyscf.dft import numint
from pyscf.lib import param
from equiv_dens.utils.grids import spherical_grid,\
    spherical_radial_sampling, treutler_atomic_radii_adjust 
import equiv_dens.utils.base as utils
from dftpy.formats import ase_io
from pyscf.dft import radi
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [63]:
from pyscf import gto, dft
from pyscf.scf import hf
hf.MUTE_CHKFILE = True

mol = gto.M(atom='H 0, 0, 0', basis='ccpvdz', spin=1)
print('nprim', mol.bas_nprim(0))
print('nctr', mol.bas_nctr(0))
print('exp', mol.bas_exp(0))
print('kappa', mol.bas_kappa(0))
print('ctr coeff', mol.bas_ctr_coeff(0))
print('env', mol._env)
print('bas', mol._bas)
print('atom', mol._atm)
print('basis', mol._basis)
print('cart', mol.cart)

nprim 3
nctr 1
exp [13.01    1.962   0.4446]
kappa 0
ctr coeff [[0.03349873]
 [0.2348008 ]
 [0.81368296]]
env [1.37036000e+02 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 1.30100000e+01 1.96200000e+00 4.44600000e-01 5.79764064e-01
 9.83419580e-01 1.11930215e+00 1.22000000e-01 5.21536727e-01
 7.27000000e-01 1.95840453e+00]
bas [[ 0  0  3  1  0 24 27  0]
 [ 0  0  1  1  0 30 31  0]
 [ 0  1  1  1  0 32 33  0]]
atom [[ 1 20  1 23  0  0]]
basis {'H': [[0, [13.01, 0.019685], [1.962, 0.137977], [0.4446, 0.478148]], [0, [0.122, 1.0]], [1, [0.727, 1.0]]]}
cart False


In [53]:
mf = dft.RKS(mol)
mf.chkfile=False
mf.xc = 'pbe'
mf.kernel()
mf.mo_occ
mf.mo_coeff.shape

converged SCF energy = -0.498628119738485  <S^2> = 0.75  2S+1 = 2


(2, 5, 5)

In [120]:

positions = np.array([0, 0, 0]).reshape(1, 1, 3)
atoms = {'atom_types': ['H'], 'positions': positions}
grid_spec = spherical_grid(atoms)

f_radii_adjust = treutler_atomic_radii_adjust([1],
                                              radi.BRAGG_RADII)
sample_coords, coord_weights = spherical_radial_sampling(grid_spec, 1000000000,
                                                ['H'],
                                                torch.DoubleTensor(positions), radii_adjust=f_radii_adjust)
coord_weights.shape

level 2


torch.Size([1, 5468])

In [121]:
scaled_sample_coords = sample_coords.detach().cpu().numpy() / param.BOHR
ao = numint.eval_ao(mol, scaled_sample_coords[0])


In [122]:
print('atomic_orbitals', ao)
print('atomic_orbitals shape', ao.shape)

atomic_orbitals [[ 7.56715260e-01  1.47122794e-01  6.12780175e-05  0.00000000e+00
   0.00000000e+00]
 [ 7.56714298e-01  1.47122788e-01  5.63037309e-04  0.00000000e+00
   0.00000000e+00]
 [ 7.56702234e-01  1.47122711e-01  2.06026490e-03  0.00000000e+00
   0.00000000e+00]
 ...
 [ 3.96411645e-02  8.32339528e-02 -3.64407281e-02 -5.80169179e-02
  -1.10367373e-02]
 [ 2.19661405e-02  7.07980564e-02 -1.57431182e-02 -2.50644607e-02
  -4.76808968e-03]
 [ 1.05291814e-02  5.78626993e-02 -5.34372270e-03 -8.50768735e-03
  -1.61844361e-03]]
atomic_orbitals shape (5468, 5)


In [12]:
dataset = AtomsDensityData(np_path=args.np_dataset, density_path=args.dens_dataset,
                           orbitals_path=args.orbitals_file,
                           density_n_samp=10000000000,
                           required_properties=required_properties,
                           center_positions=False,
                           radial_coeffs_file=args.radial_coeffs_file,
                           L0_coeffs_file=args.L0_coeffs_file,
                           dtype=args.dtype,
                           grid_fn=grid_fn,
                           sampling_fn=sampling_fn,
                           grid_extent=grid_extent,
                           grid_origin=grid_origin,
                           verbose=args.verbose
                          )



clebsch_gordan = ClebschGordanMatrix()
repr_model = EquivariantSphericalHarmonics(
    orbitals=dataset.orbitals,
    order=args.order,
    mixing_order=args.mixing_order,
    num_features=args.num_features,
    num_basis_functions=args.num_basis_functions,
    num_modules=args.num_modules,
    num_residual_pre_x=args.num_residual_pre_x,
    num_residual_post_x=args.num_residual_post_x,
    num_residual_pre_vi=args.num_residual_pre_vi,
    num_residual_pre_vj=args.num_residual_pre_vj,
    num_residual_post_v=args.num_residual_post_v,
    num_residual_output=args.num_residual_output,
    num_radial_components=args.num_radial_components,
    basis_functions=args.basis_functions,
    cutoff=args.cutoff,
    activation=args.activation,
    clebsch_gordan=clebsch_gordan,
    verbose=args.verbose,
    timing=args.timing,
)

wavefunction_model = WavefunctionCoeffsNetwork(orbitals=none,  # orbitals of atoms, defines layout and shape of output matrix
                 order=1,  # maximum order of spherical harmonics features
                 num_features=32,
                 clebsch_gordan=none,
                 verbose=0,
                 compressed_extraction=false,
                 timing=false,
                 init_coeffs=none,
                 )


expansion_model = density_expansion(dataset.orbitals, radial_coeffs=dataset.radial_coeffs,
                                    expansion_constraint=args.expansion_constraint,
                                    integral_constraint=args.integral_constraint,
                                    integral_scale=args.integral_scale,
                                    softmax_norm=args.softmax_norm, n_electrons=sum(z_vals),
                                    verbose=args.verbose,
                                    timing=args.timing,
                                    )



NameError: name 'AtomsDensityData' is not defined

In [123]:
from equiv_dens.nn.property_output.wavefunction import WavefunctionDensityExpansion
basis_def = np.load('datasets/ccpvdz_libcint_orbital_basis_df.npy', allow_pickle=True).item()
radial_coeffs = np.load('datasets/ccpvdz_libcint_radial_coeffs_df.npy', allow_pickle=True).item()

orbitals = [basis_def['H']]
coeffs = [radial_coeffs['H']]
print('orbitals', orbitals)
print('coeffs', coeffs)
print('pyscf basis', mol._basis)
print(orbitals[0])
expansion_model = WavefunctionDensityExpansion(orbitals, radial_coeffs=coeffs,
                                               n_electrons=1,
                                               verbose=3,
                                              )
print('spherical spec', expansion_model.spherical_spec)
print('radial spec', expansion_model.radial_spec)
print('rmax', expansion_model.r_max)
atoms2 = {key: atoms[key] for key in atoms.keys()}
atoms2['coords'] = scaled_sample_coords
atoms2['positions'] = torch.DoubleTensor(atoms['positions'])
atoms2 = expansion_model(atoms2)
print('atomic orbitals', atoms2['atomic_orbitals'])
print('atomic orbitals shape', atoms2['atomic_orbitals'].shape)

orbitals [[(1, 3, 0), (1, 1, 0), (1, 1, 1)]]
coeffs [[(array([13.01  ,  1.962 ,  0.4446]), array([0.57976406, 0.98341958, 1.11930215])), (array([0.122]), array([0.52153673])), (array([0.727]), array([1.95840453]))]]
pyscf basis {'H': [[0, [13.01, 0.019685], [1.962, 0.137977], [0.4446, 0.478148]], [0, [0.122, 1.0]], [1, [0.727, 1.0]]]}
[(1, 3, 0), (1, 1, 0), (1, 1, 1)]
spherical spec [[(1, 2, 0), (1, 1, 1)]]
radial spec [[(1, 4, 0), (1, 1, 1)]]
rmax {(1, 0): 3, (1, 1): 1}
width shape torch.Size([1, 1, 3, 2])
rbf shape torch.Size([1, 5468, 1, 2])
sph shape torch.Size([1, 5468, 1, 1])
width shape torch.Size([1, 1, 1, 1])
rbf shape torch.Size([1, 5468, 1, 1])
sph shape torch.Size([1, 5468, 3, 1])
atomic orbitals tensor([[[ 7.5672e-01,  1.4712e-01, -9.5688e-01,  0.0000e+00,  0.0000e+00],
         [ 7.5671e-01,  1.4712e-01, -9.5688e-01,  0.0000e+00,  0.0000e+00],
         [ 7.5670e-01,  1.4712e-01, -9.5688e-01,  0.0000e+00,  0.0000e+00],
         ...,
         [ 3.9641e-02,  8.3234e-02,  1.6

In [124]:
print('bas exp 0', mol.bas_exp(0))
print('bas exp 0', mol.bas_exp(1))
print('bas exp 0', mol.bas_exp(2))
print('bas ctr coeff 0', mol.bas_ctr_coeff(0))
print('basis', mol._basis)
print('radial_coeffs', radial_coeffs['H'])
print('basis', mol._env)

bas exp 0 [13.01    1.962   0.4446]
bas exp 0 [0.122]
bas exp 0 [0.727]
bas ctr coeff 0 [[0.03349873]
 [0.2348008 ]
 [0.81368296]]
basis {'H': [[0, [13.01, 0.019685], [1.962, 0.137977], [0.4446, 0.478148]], [0, [0.122, 1.0]], [1, [0.727, 1.0]]]}
radial_coeffs [(array([13.01  ,  1.962 ,  0.4446]), array([0.57976406, 0.98341958, 1.11930215])), (array([0.122]), array([0.52153673])), (array([0.727]), array([1.95840453]))]
basis [1.37036000e+02 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 1.30100000e+01 1.96200000e+00 4.44600000e-01 5.79764064e-01
 9.83419580e-01 1.11930215e+00 1.22000000e-01 5.21536727e-01
 7.27000000e-01 1.95840453e+00]


In [125]:
print('expansion ao', atoms2['atomic_orbitals'])
print('pyscf ao', ao)
ao_t = torch.DoubleTensor(ao)

print('expansion ao integral', torch.sum(atoms2['atomic_orbitals'].squeeze() * coord_weights.view(-1, 1), dim=0))
print('pyscf ao integral', torch.sum(ao_t * coord_weights.view(-1, 1), dim=0))

print('ratio', atoms2['atomic_orbitals'] / ao_t)

expansion ao tensor([[[ 7.5672e-01,  1.4712e-01, -9.5688e-01,  0.0000e+00,  0.0000e+00],
         [ 7.5671e-01,  1.4712e-01, -9.5688e-01,  0.0000e+00,  0.0000e+00],
         [ 7.5670e-01,  1.4712e-01, -9.5688e-01,  0.0000e+00,  0.0000e+00],
         ...,
         [ 3.9641e-02,  8.3234e-02,  1.6865e-02,  2.6850e-02,  5.1078e-03],
         [ 2.1966e-02,  7.0798e-02,  6.4296e-03,  1.0236e-02,  1.9473e-03],
         [ 1.0529e-02,  5.7863e-02,  1.9321e-03,  3.0761e-03,  5.8518e-04]]],
       dtype=torch.float64)
pyscf ao [[ 7.56715260e-01  1.47122794e-01  6.12780175e-05  0.00000000e+00
   0.00000000e+00]
 [ 7.56714298e-01  1.47122788e-01  5.63037309e-04  0.00000000e+00
   0.00000000e+00]
 [ 7.56702234e-01  1.47122711e-01  2.06026490e-03  0.00000000e+00
   0.00000000e+00]
 ...
 [ 3.96411645e-02  8.32339528e-02 -3.64407281e-02 -5.80169179e-02
  -1.10367373e-02]
 [ 2.19661405e-02  7.07980564e-02 -1.57431182e-02 -2.50644607e-02
  -4.76808968e-03]
 [ 1.05291814e-02  5.78626993e-02 -5.34372270e-0

1.7017388874777748
1.7017386955796983
1.7017387085170281


In [14]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
%matplotlib notebook
sample = 5
fig = plt.figure(figsize=(10, 5))
ax1 = fig.add_subplot(131)
dens = np.array(valid_cube_dataset[sample]['density'].squeeze())
print('dens shape', dens.shape)
x, y  = np.meshgrid(np.arange(50), np.arange(50))
x_f = x.flatten()
y_f = y.flatten()
z_f = np.zeros_like(y_f) + 19
idx = np.ravel_multi_index((x_f, y_f, z_f), (50, 50, 50))
                
#X, Y, Z = np.meshgrid(np.arange(50) + 38, np.arange(50) + 38, np.arange(50) + 38)
#X = X.flatten()
#Y = Y.flatten()
#Z = Z.flatten()
#ndices = np.ravel_multi_index((X, Y, Z), (125, 125, 125))

print('idx', idx)
dens2d = dens[idx]
dens2d = dens2d.reshape(50, 50)

indices = np.stack((x_f, y_f, z_f), axis=1)
coords_space = valid_cube_dataset[sample]['coords']
print('coords space shape', coords_space.shape)
vmin = np.min(dens2d)
vmax = np.max(dens2d)
c1 = ax1.contourf(x, y, dens2d, vmin=vmin, vmax=vmax, levels=100)
ax1.set_aspect('equal')

data_sample = valid_cube_dataset[sample]
pred_dens = model(data_sample)



print(pred_dens['density'].shape)

p_dens = pred_dens['density'].cpu().data.squeeze().numpy()
p_dens = p_dens[idx].reshape(50, 50)

ax2 = fig.add_subplot(132)
c2 = ax2.contourf(x, y, p_dens, vmin=vmin, vmax=vmax, levels=100)
ax2.set_aspect('equal')

ax3 = fig.add_subplot(133)
c3 = ax3.contourf(x, y, p_dens-dens2d, vmin=-0.1, vmax=0.1, levels=100)
ax3.set_aspect('equal')

fig.colorbar(c2, ax=[ax1, ax2])
fig.colorbar(c3, ax=[ax3])
plt.savefig('figures/target_vs_pred_dens.pdf', dpi=300)
plt.show()

<IPython.core.display.Javascript object>

NameError: name 'valid_cube_dataset' is not defined